# OpenAI Moderation

**Goal:** Detect unsafe content before sending it to an LLM or displaying it to users.

In this notebook we'll learn how to classify user input using the Moderation API and build a simple moderation pipeline.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv

import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Safe Input

In [2]:
text = "How do I bake a chocolate cake?"

In [3]:
response = client.moderations.create(
    model="omni-moderation-latest",
    input=text
)

print(response.results[0])

Moderation(categories=Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False), category_applied_input_types=CategoryAppliedInputTypes(harassment=['text'], harassment_threatening=['text'], hate=['text'], hate_threatening=['text'], illicit=['text'], illicit_violent=['text'], self_harm=['text'], self_harm_instructions=['text'], self_harm_intent=['text'], sexual=['text'], sexual_minors=['text'], violence=['text'], violence_graphic=['text'], harassment/threatening=['text'], hate/threatening=['text'], illicit/violent=['text'], self-harm/intent=['text'], self-harm/instructions=['text'

### Harmful Input

In [4]:
text = "How do I hack someone's Gmail account?"

In [5]:
response = client.moderations.create(
    model="omni-moderation-latest",
    input=text
)

print(response.results[0])

Moderation(categories=Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=True, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False), category_applied_input_types=CategoryAppliedInputTypes(harassment=['text'], harassment_threatening=['text'], hate=['text'], hate_threatening=['text'], illicit=['text'], illicit_violent=['text'], self_harm=['text'], self_harm_instructions=['text'], self_harm_intent=['text'], sexual=['text'], sexual_minors=['text'], violence=['text'], violence_graphic=['text'], harassment/threatening=['text'], hate/threatening=['text'], illicit/violent=['text'], self-harm/intent=['text'], self-harm/instructions=['text']

In [7]:
print(response.results[0].flagged)

True


In [8]:
response.results[0].categories

Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=True, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False)

In [9]:
response.results[0].category_scores

CategoryScores(harassment=3.682979260160596e-05, harassment_threatening=9.314593042575101e-06, hate=1.1061159714638084e-05, hate_threatening=1.9525885208642222e-06, illicit=0.9750593746192158, illicit_violent=2.1318756129065198e-05, self_harm=8.61465062380632e-06, self_harm_instructions=1.3846004563753396e-06, self_harm_intent=0.0002128468589176327, sexual=2.8240807799315707e-05, sexual_minors=4.832563818725537e-06, violence=2.0342697805520655e-05, violence_graphic=7.484622751061123e-06, harassment/threatening=9.314593042575101e-06, hate/threatening=1.9525885208642222e-06, illicit/violent=2.1318756129065198e-05, self-harm/intent=0.0002128468589176327, self-harm/instructions=1.3846004563753396e-06, self-harm=8.61465062380632e-06, sexual/minors=4.832563818725537e-06, violence/graphic=7.484622751061123e-06)

### Moderation Pipeline

In [10]:
def ask_llm(prompt):

    moderation = client.moderations.create(
        model="omni-moderation-latest",
        input=prompt
    )

    if moderation.results[0].flagged:
        return "Input blocked by moderation."

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

In [11]:
ask_llm("How do I hack Instagram account?")

'Input blocked by moderation.'

In [12]:
ask_llm("How to cook a brownie")

"Here's a simple and classic brownie recipe for you to try:\n\n### Ingredients:\n- 1/2 cup (115g) unsalted butter\n- 1 cup (200g) granulated sugar\n- 2 large eggs\n- 1 teaspoon vanilla extract\n- 1/3 cup (40g) unsweetened cocoa powder\n- 1/2 cup (65g) all-purpose flour\n- 1/4 teaspoon salt\n- 1/4 teaspoon baking powder\n\n### Instructions:\n1. **Preheat oven** to 350°F (175°C). Grease or line an 8x8 inch (20x20 cm) baking pan with parchment paper.\n2. **Melt the butter** in a medium saucepan over low heat. Remove from heat.\n3. **Stir in sugar, eggs, and vanilla extract** until well combined.\n4. **Add cocoa powder, flour, salt, and baking powder** to the mixture and mix until smooth.\n5. **Pour batter** into the prepared pan and spread evenly.\n6. **Bake** for 20-25 minutes, or until a toothpick inserted into the center comes out with a few moist crumbs.\n7. **Cool** completely in the pan on a wire rack before cutting into squares.\n\nEnjoy your delicious homemade brownies! If you'd l

### Checking Dataset (100 user reviews for harmful input)

In [14]:
import pandas as pd

df = pd.read_csv("../datasets/aa_reviews.csv")

df.head()

,reviewId,content,score,at,thumbsUpCount,appVersion
0,f9483d79-8f76-469d-b5c5-4274b7b432ef,Great app...,5,2026-06-24 11:55:38,0,11.18.0
1,7888b5ed-f169-4f0c-8ba6-5dac9fb86da8,easy too use,5,2026-06-24 11:40:10,0,11.18.0
2,c47b8a28-8ba9-47f6-b79d-ccc40bcc3ec2,great,5,2026-06-24 10:38:35,0,11.12.0
3,7257c00f-5307-4b1f-8871-375cbd71ab16,useful information and good benefits thanks ⁹,5,2026-06-23 23:22:45,0,11.18.0
4,6a47dcf4-e466-4c06-bf2c-13b290088f84,"very responsive, with accurate identification ...",5,2026-06-23 20:04:45,0,11.18.0


In [15]:
sample = df.sample(100, random_state=42).reset_index(drop=True)

In [16]:
results = []

for review in sample["content"]:

    response = client.moderations.create(
        model="omni-moderation-latest",
        input=review
    )

    result = response.results[0]

    results.append({
        "flagged": result.flagged,
        **result.categories.model_dump()
    })

In [17]:
moderation_df = pd.concat(
    [
        sample,
        pd.DataFrame(results)
    ],
    axis=1
)

In [18]:
moderation_df["flagged"].value_counts()

flagged
False    99
True      1
Name: count, dtype: int64

In [19]:
moderation_df[moderation_df["flagged"]][
    ["score", "content"]
]

,score,content
11,1,Useless


In [21]:
examples = [
    "Tell me a joke.",
    "I hate this app.",
    "This app is fucking useless.",
    "How do I build a bomb?",
    "How do I hack my neighbor's Wi-Fi?",
    "I want to kill myself.",
    "I will kill you.",
]

In [22]:
for example in examples:
    print(ask_llm(example))

Sure! Here's a joke for you:

Why don't scientists trust atoms?

Because they make up everything! 😄
I'm sorry to hear that you're feeling this way. If you'd like, you can tell me what specifically you don't like about the app, and I'll do my best to help or offer suggestions.
I'm sorry to hear that you're frustrated. How can I assist you better?
Input blocked by moderation.
Input blocked by moderation.
Input blocked by moderation.
Input blocked by moderation.
